<a href="https://colab.research.google.com/github/ironspiritjeff/hf-transformer-experiments/blob/main/Object_Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Object Detection with Hugging Face Transformers

In [ ]:
# Install / import dependencies
try:
  import datasets
  import gradio as gr
  import torchmetrics
  import pycocotools
except ModuleNotFoundError:
  !pip install -U datasets gradio
  !pip install -U torchmetrics[detection]

  import datasets
  import gradio as gr
  import torchmetrics
  import pycocotools

import random
import numpy as np
import torch
import transformers
from pprint import pprint

# Check versions
print(f"Using transformers version: {transformers.__version__}")
print(f"Using datasets version: {datasets.__version__}")
print(f"Using torch version: {torch.__version__}")
print(f"Using torchmetrics version: {torchmetrics.__version__}")

## Getting a Dataset

In [ ]:
from datasets import load_dataset

dataset = load_dataset(path="mrdbourke/trashify_manual_labelled_images")

dataset

In [ ]:
pprint(dataset["train"][0])

In [ ]:
#for i in range(10):
  #display(dataset['train'][i]['image'])
  #print(f'image: {i+1} of 10')

In [ ]:
from pprint import pprint
pprint(dataset["train"].features)

In [ ]:
categories = dataset['train'].features['annotations']['category_id'].feature

categories.names

## Map numbers to labels

In [ ]:
# Map ID values to class names and vice versa
id2label = {i: class_name for i,
            class_name in enumerate(categories.names)}
label2id = {value: key for key,
            value in id2label.items()}
id2label, label2id

Create a colour palette

In [ ]:
# Make a color dictionary = mapping from class name to RGB color
colour_palette = {
    'bin': (0, 0, 224),         # Bright Blue (High contrast with greenery) in format (red, green, blue)
    'not_bin': (255, 80, 80),   # Light Red to indicate negative class

    'hand': (148, 0, 211),      # Dark Purple (Contrasts well with skin tones)
    'not_hand': (255, 80, 80),  # Light Red to indicate negative class

    'trash': (0, 255, 0),       # Bright Green (For trash-related items)
    'not_trash': (255, 80, 80), # Light Red to indicate negative class

    'trash_arm': (255, 140, 0), # Deep Orange (Highly visible)
}

In [ ]:
import matplotlib.pyplot as plt

# Normalize RGB values to 0-1 range
def normalize_rgb(rgb_tuple):
  return tuple(x/255.0 for x in rgb_tuple)

# Turn colors into normalized RGB values for matplotlib
colors_and_labels_rgb = [(key, normalize_rgb(value)) for key, value in colour_palette.items()]
colors_and_labels_rgb

# Create a figure:
fig, ax = plt.subplots(nrows=1, ncols=7, figsize=(8,1))

# Flatten the axes
ax = ax.flatten()

# Plot each color on a square
for idx, (label, color) in enumerate(colors_and_labels_rgb):
  ax[idx].add_patch(plt.Rectangle(xy=(0,0),
                                  width=1,
                                  height=1,
                                  facecolor=color))
  ax[idx].set_title(label)
  ax[idx].axis('off')

plt.tight_layout()
plt.show()

## Plotting a single image and visualize boxes

### Create helper functions to first view images and boxes

In [ ]:
import PIL

def half_image(image: PIL.Image) -> PIL.Image:
  '''
  Resizes a given input image by half and returns the smaller version.
  '''
  return image.resize(size=(image.size[0] // 2, image.size[1] // 2))


In [ ]:
def quarter_image(image: PIL.Image) -> PIL.Image:
  '''
  Resizes a given input image by 1/4th and returns the smaller version.
  '''
  return image.resize(size=(image.size[0] // 4, image.size[1] // 4))

In [ ]:
quarter_image(dataset['train'][7]['image'])

In [ ]:
def half_boxes(boxes):
  '''
  Halves an array of input boxes and returns them, resized for half images.
  '''
  # Work with lists, or lists of lists:
  if isinstance(boxes, list):
    # If boxes are list of lists, then we have multiple boxes
    for box in boxes:
      if isinstance(box, list):
        return[[coordinate // 2 for coordinate in box] for box in boxes]
      else:
        return[coordinate // 2 for coordinate in boxes]

  # Work with Numpy arrays
  if isinstance(boxes, np.ndarray):
    return (boxes // 2)

  # Work with Torch tensors:
  if isinstance(boxes, torch.Tensor):
    return (boxes // 2)

In [ ]:
# Test half_boxes and half_image helper functions:
print("half_boxes list function:")
print(dataset['train'][7]['annotations']['bbox'])
print(half_boxes(dataset['train'][7]['annotations']['bbox']))
print(f"half_boxes torch function: {half_boxes(torch.tensor([[100,100,100,100], [200,200,200,200]]))}")

print("half_image function:")
print(dataset['train'][7]['image'].size)
print(half_image(dataset['train'][7]['image']).size)

### Plotting a single image with bounding boxes, step by step:

Working towards 100 samples
1. Select a random sample
2. Get the image from `image`
* Optional: half the image/boxes
3. Turn box coordinates into a torch tensor
(in order to use 'torchvision' to help plot boxes)
4. Convert bounding box format from `XYWH` to  `XYXY`, using `torchvision.ops.box_convert` (because `XYXY` is required by `torchvision.utils.draw_bounding_boxes`)
5. Get a list of label names from the `category_id` field, in ordre to plot the class names of the particular box on the rectangle.
6. Draw the target boxes onto the target image:
 * Turn PIL image into tensor. (`torchvision.transformers.functional.pil_to_tensor`)
 * Draw boxes (`torchvision.utils.draw_bounding_boxes`)
 * Turn image and drawn boxes from tensor back to `PIL` image (`torchvision.transformers.functional.tensor_to_pil`)

In [ ]:
import random

random_idx =
random_image =

bbox_coordinates_xywh =
bbox_coordinates_xyxy =

list_of_class_names =

drawn_boxes =